In [ ]:
!pip install -U transformers datasets accelerate peft trl bitsandbytes scikit-learn pandas numpy

In [ ]:
!pip install -U seqeval

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import pandas as pd
import numpy as np
import json
import ast
from sklearn.model_selection import train_test_split
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from transformers import pipeline
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, precision_score, recall_score, lassification_report
import evaluate

In [ ]:
FULL_DATA_PATH = "/content/drive/MyDrive/term_paper_masha/bank_reviews_dataset.csv"
MANUAL_LABELS_PATH = "/content/drive/MyDrive/term_paper_masha/label_studio_with_additional.csv"
UNLABELED_PATH = "/content/drive/MyDrive/term_paper_masha/bank_reviews_dataset_without_manual.csv"

In [ ]:
df = pd.read_csv(MANUAL_LABELS_PATH)
df.head()

In [ ]:
df = df.dropna(subset=["text", "label"]).copy()
df = df.reset_index(drop=True)

def parse_label_cell(x):
    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    try:
        return json.loads(x)
    except:
        try:
            return ast.literal_eval(x)
        except:
            return []

df["spans"] = df["label"].apply(parse_label_cell)
df[["id", "text", "spans"]].head()

In [ ]:
span_rows = []

for _, row in df.iterrows():
    review_id = row["id"]
    full_text = row["text"]

    for span in row["spans"]:
        labels = span.get("labels", [])

        for label in labels:
            span_rows.append({
                "review_id": review_id,
                "full_text": full_text,
                "evidence": span.get("text", ""),
                "label": label,
                "start": span.get("start"),
                "end": span.get("end"),
                "rating": row.get("rating"),
                "source": row.get("source"),
                "title": row.get("title")
            })

spans_df = pd.DataFrame(span_rows)
spans_df.head(20)

In [ ]:
spans_df.to_csv("absa_spans.csv", index=False, encoding="utf-8-sig")

In [ ]:
teacher_rows = []

for _, row in df.iterrows():
    aspects = []

    for span in row["spans"]:
        labels = span.get("labels", [])

        for label in labels:
            aspects.append({
                "label": label,
                "evidence": span.get("text", "")
            })

    teacher_rows.append({
        "id": row["id"],
        "text": row["text"],
        "aspects": aspects
    })

teacher_df = pd.DataFrame(teacher_rows)
teacher_df.head()

In [ ]:
teacher_df.to_json("manual_teacher_format.json",orient="records",force_ascii=False,indent=2)

In [ ]:
LABELS = [
    "CARDS_POS", "CARDS_NEG",
    "DIGITAL_POS", "DIGITAL_NEG",
    "SUPPORT_POS", "SUPPORT_NEG",
    "PAYMENTS_POS", "PAYMENTS_NEG",
    "CREDITS_POS", "CREDITS_NEG",
    "FEES_POS", "FEES_NEG",
    "OFFICE_POS", "OFFICE_NEG",
    "ATM_POS", "ATM_NEG",
    "FRAUD_POS", "FRAUD_NEG",
    "INSURANCE_POS", "INSURANCE_NEG",
    "INVESTMENTS_POS", "INVESTMENTS_NEG",
    "PREMIUM_POS", "PREMIUM_NEG",
    "OTHER_POS", "OTHER_NEG"
]

label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

In [ ]:
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

print(label2id)

In [ ]:
manual_train_val, manual_test = train_test_split(
    teacher_df,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

manual_train, manual_val = train_test_split(
    manual_train_val,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

print("manual_train:", len(manual_train))
print("manual_val:", len(manual_val))
print("manual_test:", len(manual_test))

In [ ]:
def build_teacher_prompt(text):
    return f"""
Ты размечаешь отзывы клиентов банка для задачи Aspect-Based Sentiment Analysis.

Нужно найти в отзыве фрагменты текста, которые относятся к банковским аспектам,
и определить тональность каждого аспекта.

Разрешенные labels:
{LABELS}

Верни только JSON формата:
{{
  "aspects": [
    {{
      "label": "SUPPORT_NEG",
      "evidence": "фрагмент текста"
    }}
  ]
}}

Если аспектов нет, верни:
{{"aspects": []}}

Текст отзыва:
{text}
""".strip()


def aspects_to_json(aspects):
    return json.dumps({"aspects": aspects},ensure_ascii=False)

In [ ]:
teacher_train_rows = []

for _, row in manual_train.iterrows():
    teacher_train_rows.append({
        "messages": [
            {
                "role": "user",
                "content": build_teacher_prompt(row["text"])
            },
            {
                "role": "assistant",
                "content": aspects_to_json(row["aspects"])
            }
        ]
    })

with open("teacher_train.jsonl", "w", encoding="utf-8") as f:
    for item in teacher_train_rows:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [ ]:
TEACHER_MODEL = "Qwen/Qwen3-8B"
STUDENT_MODEL = "deepvk/RuModernBERT-base"

In [ ]:
dataset_teacher = load_dataset("json",data_files="teacher_train.jsonl", split="train")

tokenizer_teacher = AutoTokenizer.from_pretrained(TEACHER_MODEL, trust_remote_code=True)

bnb_config = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_compute_dtype=torch.bfloat16,
bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True)

teacher_model = AutoModelForCausalLM.from_pretrained(TEACHER_MODEL,quantization_config=bnb_config,
device_map="auto", trust_remote_code=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [ ]:
def formatting_func(example):
    return tokenizer_teacher.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

sft_config = SFTConfig(
    output_dir="./qwen_teacher_absa",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=100,
    max_seq_length=2048,
    bf16=True,
    report_to="none"
)

trainer_teacher = SFTTrainer(
    model=teacher_model,
    args=sft_config,
    train_dataset=dataset_teacher,
    peft_config=lora_config,
    formatting_func=formatting_func
)

trainer_teacher.train()
trainer_teacher.save_model("./qwen_teacher_absa")

In [ ]:
unlabeled_df = pd.read_csv(UNLABELED_PATH)

unlabeled_df.head()

In [ ]:
teacher_pipe = pipeline("text-generation", model=teacher_model, tokenizer=tokenizer_teacher)

In [ ]:
def generate_teacher_markup(text, max_new_tokens=512):
    messages = [
        {
            "role": "user",
            "content": build_teacher_prompt(text)
        }
    ]
    prompt = tokenizer_teacher.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    output = teacher_pipe(prompt, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0, return_full_text=False)[0]["generated_text"]

    return output.strip()

In [ ]:
teacher_rows = []

for i, row in unlabeled_df.iterrows():
    text = row["text"]
    try:
        answer = generate_teacher_markup(text)
        teacher_rows.append({
            "text": text,
            "teacher_answer": answer
        })
    except Exception as e:
        teacher_rows.append({
            "text": text,
            "teacher_answer": None,
            "error": str(e)
        })
    if i % 50 == 0:
        print("processed:", i)

by_teacher_df = pd.DataFrame(teacher_rows)
by_teacher_df.to_csv("labeled_by_teacher.csv", index=False, encoding="utf-8-sig")